# TSO-DTF Decision Dashboard (Google Colab)

**Project:** Decision Tree Optimization Framework for Advanced Decision Support and Business Analytics Applications

Builds an interactive business dashboard from the saved forecasts (no model training):
1. **Sales overview** - weekly revenue/units, departments, demand types
2. **Demand forecast** - TSO-DTF vs base paper DTF vs actual, product drill-down
3. **Inventory planning** - service target, stock value, lost sales (order-up-to simulation)
4. **Order plan** - how many units to order per product for the next day
5. **Model accuracy** - M5 levels, metrics, ablation, bootstrap interval

**Needs in Drive:** `m5_data` (calendar, sell_prices, sales_train_evaluation) and in `m5_results`: `baseline_predictions.csv.gz`,
`optimization4/5/6/7/10_predictions.csv.gz`, `eval_products.csv` (same files as the results notebook).

**Run:** Runtime -> Run all (about 5-10 minutes). The dashboard appears at the bottom and is saved as `m5_results/dashboard.html`
(download it and open in any browser).

In [1]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/m5_data"
RES_DIR = "/content/drive/MyDrive/m5_results"


Mounted at /content/drive


## Step 1 - Load data and forecasts (final TSO-DTF = average of Opt 4, 5, 6, 7, 10)

In [2]:
import os, glob, json, html, numpy as np, pandas as pd
from scipy.stats import wilcoxon, norm
def find(name):
    hits = glob.glob(f"{RES_DIR}/**/{name}", recursive=True)
    if not hits: raise FileNotFoundError(f"{name} not found in {RES_DIR}")
    return hits[0]

pp = pd.read_csv(find("eval_products.csv"))                     # the 9,119 scored series (same for every method)
cal = pd.read_csv(f"{DATA_DIR}/calendar.csv", parse_dates=["date"])
sales = pd.concat([ch[ch["store_id"].isin(["CA_1", "TX_1", "WI_1"])] for ch in pd.read_csv(f"{DATA_DIR}/sales_train_evaluation.csv", chunksize=3000)])
sales = pp[["store_id", "item_id"]].merge(sales, on=["store_id", "item_id"], how="left")
dcols = [c for c in sales.columns if c.startswith("d_")]
Y = sales[dcols].values.astype(float); N = len(pp)
T0 = int(np.where(cal["date"].values[:len(dcols)] == np.datetime64("2016-03-22"))[0][0]); T = 62
A = Y[:, T0:T0 + T]                                             # actual sales in the test period
print(f"{N:,} series | test {cal.date.iloc[T0].date()} to {cal.date.iloc[T0 + T - 1].date()} ({T} days)")

def load_pred(fname):
    p = pd.read_csv(find(fname), parse_dates=["date"])
    p = pp[["store_id", "item_id"]].reset_index().merge(p, on=["store_id", "item_id"]).sort_values(["index", "date"])
    assert len(p) == N * T, f"{fname}: {len(p)} rows, expected {N * T}"
    assert np.allclose(p["sales"].values.reshape(N, T), A), f"{fname}: actuals do not match"
    return np.maximum(p["pred"].values.reshape(N, T), 0)
F = {"Baseline DTF": load_pred("baseline_predictions.csv.gz"),
     "Opt 4": load_pred("optimization4_predictions.csv.gz"), "Opt 5": load_pred("optimization5_predictions.csv.gz"),
     "Opt 6": load_pred("optimization6_predictions.csv.gz"), "Opt 7": load_pred("optimization7_predictions.csv.gz"),
     "Opt 10": load_pred("optimization10_predictions.csv.gz")}
F["Ensemble 4+5"] = (F["Opt 4"] + F["Opt 5"]) / 2
F["TSO-DTF (final)"] = (F["Opt 4"] + F["Opt 5"] + F["Opt 6"] + F["Opt 7"] + F["Opt 10"]) / 5   # equal-weight ensemble
print("Loaded:", list(F))


9,119 series | test 2016-03-22 to 2016-05-22 (62 days)
Loaded: ['Baseline DTF', 'Opt 4', 'Opt 5', 'Opt 6', 'Opt 7', 'Opt 10', 'Ensemble 4+5', 'TSO-DTF (final)']


## Step 2 - Model accuracy (official M5 WRMSSE, product metrics, ablation, bootstrap)

In [3]:
first = np.argmax(Y[:, :T0] > 0, axis=1)
scale = np.array([np.mean(np.diff(Y[i, first[i]:T0]) ** 2) for i in range(N)])          # RMSSE scale (from first sale)
mad = np.mean(np.abs(np.diff(Y[:, T0 - 365:T0], axis=1)), axis=1)                         # SME scale (365-day history)
w = pp["weight"].values
def product_metrics(P):
    e = A - P; r = np.sqrt(np.mean(e ** 2, axis=1) / scale)
    return {"WRMSSE (product level)": (w * r).sum() / w.sum(), "RMSSE mean (Eq. 1)": r.mean(), "RMSSE median": np.median(r),
            "RMSE": np.sqrt((e ** 2).mean()), "MAE": np.abs(e).mean(), "WAPE": np.abs(e).sum() / A.sum(),
            "SME bias (Eq. 2)": np.nanmean(np.mean(e, axis=1) / mad)}, r

# ---- official M5 multi-level WRMSSE ----
prices = pd.concat([ch[ch["store_id"].isin(["CA_1", "TX_1", "WI_1"])] for ch in pd.read_csv(f"{DATA_DIR}/sell_prices.csv", chunksize=500000)])
pm = prices.pivot_table(index=["store_id", "item_id"], columns="wm_yr_wk", values="sell_price").reindex(pd.MultiIndex.from_frame(pp[["store_id", "item_id"]]))
wk = cal["wm_yr_wk"].values[:len(dcols)]
P28 = pm.reindex(columns=wk[T0 - 28:T0]).values
rev28 = np.nansum(Y[:, T0 - 28:T0] * np.nan_to_num(P28), axis=1)                           # weights: revenue 28 days before test
dept = np.array([i.rsplit("_", 1)[0] for i in pp.item_id]); catg = np.array([i.split("_")[0] for i in pp.item_id]); st = pp.store_id.values
LEVELS = {"Total": np.zeros(N, int), "Store": st, "Category": catg, "Department": dept, "Store-category": np.char.add(st, catg),
          "Store-department": np.char.add(st, dept), "Product (all stores)": pp.item_id.values, "Product-store": np.arange(N)}
OFFICIAL_12 = ["Total", "Store", "Store", "Category", "Department", "Store-category", "Store-category",
               "Store-department", "Store-department", "Product (all stores)", "Product-store", "Product-store"]  # 1 store per state
PRE = {}
for lv, g in LEVELS.items():
    codes = pd.Series(g).astype("category").cat.codes.values; K = codes.max() + 1
    agg = lambda M, codes=codes, K=K: np.vstack([np.bincount(codes, weights=M[:, t], minlength=K) for t in range(M.shape[1])]).T
    Yh = agg(Y[:, :T0]); sc = np.array([np.mean(np.diff(y[np.argmax(y > 0):]) ** 2) for y in Yh])
    wl = np.bincount(codes, weights=rev28, minlength=K); ok = sc > 0
    PRE[lv] = (codes, K, agg, agg(A)[ok], sc[ok], wl[ok] / wl[ok].sum(), ok)
def level_wrmsse(P, lv, days=slice(None)):
    codes, K, agg, Aa, sc, wl, ok = PRE[lv]; Pa = agg(P)[ok]
    return (wl * np.sqrt(np.mean((Aa[:, days] - Pa[:, days]) ** 2, axis=1) / sc)).sum()
def official(P, days=slice(None)):
    return np.mean([level_wrmsse(P, lv, days) for lv in OFFICIAL_12])

B, E = F["Baseline DTF"], F["TSO-DTF (final)"]
mb, rb = product_metrics(B); me, re_ = product_metrics(E)
pct_impr = lambda b, e: 100 * (b - e) / b
levels = [[lv, level_wrmsse(B, lv), level_wrmsse(E, lv)] for lv in LEVELS] + [["Official M5 score (12 levels)", official(B), official(E)]]
levels = [r + [pct_impr(r[1], r[2])] for r in levels]
metrics = [[k, mb[k], me[k], 100 * (abs(mb[k]) - abs(me[k])) / abs(mb[k]) if k.startswith("SME") else pct_impr(mb[k], me[k])] for k in mb]
abl = []
for name in ["Baseline DTF", "Opt 4", "Opt 5", "Opt 6", "Opt 7", "Opt 10", "Ensemble 4+5", "TSO-DTF (final)"]:
    pw, ow = product_metrics(F[name])[0]["WRMSSE (product level)"], official(F[name])
    abl.append([name.replace("Baseline DTF", "Base paper DTF"), pw, pct_impr(mb["WRMSSE (product level)"], pw), ow, pct_impr(levels[-1][1], ow)])
rng = np.random.default_rng(0); blocks = [np.arange(i, min(i + 7, T)) for i in range(0, T, 7)]
boot = []
for _ in range(1000):                                            # 7-day block bootstrap of the official score
    d = np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))])
    boot.append(pct_impr(official(B, d), official(E, d)))
MODEL = {"levels": levels, "metrics": metrics, "abl": abl,
         "ci": [np.percentile(boot, 2.5), np.percentile(boot, 97.5), 100 * np.mean(np.array(boot) > 1)],
         "improved": 100 * np.mean(re_ < rb), "p": float(wilcoxon(re_, rb).pvalue), "acc": [1 - mb["WAPE"], 1 - me["WAPE"]]}
print(pd.DataFrame(levels, columns=["Level", "Base paper DTF", "TSO-DTF", "Improvement %"]).round(4).to_string(index=False))
print(f"\n95% CI [{MODEL['ci'][0]:.2f}%, {MODEL['ci'][1]:.2f}%] | products improved {MODEL['improved']:.1f}% | accuracy {MODEL['acc'][0]:.2%} -> {MODEL['acc'][1]:.2%}")


                        Level  Base paper DTF  TSO-DTF  Improvement %
                        Total          0.3847   0.3602         6.3769
                        Store          0.4407   0.4273         3.0368
                     Category          0.4314   0.4048         6.1669
                   Department          0.5002   0.4625         7.5384
               Store-category          0.5069   0.4920         2.9502
             Store-department          0.5720   0.5511         3.6680
         Product (all stores)          0.7653   0.7579         0.9671
                Product-store          0.7775   0.7722         0.6786
Official M5 score (12 levels)          0.5563   0.5392         3.0773

95% CI [0.67%, 4.92%] | products improved 64.2% | accuracy 29.07% -> 29.63%


## Step 3 - Business data: sales history, forecast vs actual, inventory simulation, order plan

In [4]:
STORES = ["CA_1", "TX_1", "WI_1"]; dtype = pp["category"].values           # demand type (smooth / erratic / intermittent / lumpy)
info = pp[["store_id", "item_id"]].assign(dept_id=dept, cat_id=catg, category=dtype)
dates = cal["date"].values[:len(dcols)]
# ---- A. weekly sales (Jan 2015 to end of test) and B. department revenue (52 weeks before test) ----
H0 = int(np.where(dates == np.datetime64("2015-01-03"))[0][0])               # a Saturday = Walmart week start
PR = pm.reindex(columns=wk[H0:T0 + T]).values                               # price of each product on each day
U = Y[:, H0:T0 + T]; REV = U * np.nan_to_num(PR)
wks = wk[H0:T0 + T]; uw = [w for w in pd.unique(wks) if (wks == w).sum() == 7]   # full weeks only
grp = info.groupby(["store_id", "cat_id", "category"]).indices
wk_rows = []
for wi, w in enumerate(uw):
    cols = np.where(wks == w)[0]
    for (s, c, ty), ix in grp.items():
        wk_rows.append([wi, s, c, ty, int(U[np.ix_(ix, cols)].sum()), round(float(REV[np.ix_(ix, cols)].sum()), 1)])
week_start = [str(pd.Timestamp(dates[H0 + np.where(wks == w)[0][0]]).date()) for w in uw]
l52 = slice(T0 - 364 - H0, T0 - H0)
dept_rows = [[d, s, ty, int(U[ix, l52].sum()), round(float(REV[ix, l52].sum()), 1)]
             for (d, s, ty), ix in info.groupby(["dept_id", "store_id", "category"]).indices.items()]
price = pm.ffill(axis=1).iloc[:, -1].values; price = np.where(np.isnan(price), np.nanmedian(price), price)   # latest price
# ---- C. forecasts compared: TSO-DTF, base paper DTF, 7-day moving average ----
cs = np.concatenate([np.zeros((N, 1)), np.cumsum(Y, axis=1)], axis=1)
MA7 = (cs[:, T0:T0 + T] - cs[:, T0 - 7:T0 + T - 7]) / 7
FC = {"tso": E, "base": B, "ma7": MA7}
fva = []
for (s, c, ty), ix in grp.items():
    for t in range(T):
        fva.append([t, s, c, ty, round(float(A[ix, t].sum()), 1), round(float(E[ix, t].sum()), 1), round(float(B[ix, t].sum()), 1), round(float(MA7[ix, t].sum()), 1)])
# ---- D. inventory simulation: daily order-up-to, next-morning delivery, lost sales ----
WARM = 14; SLS = [0.80, 0.85, 0.90, 0.95, 0.975, 0.99]
def simulate(Fc, z, upto=T):
    err = A - Fc; inv = np.zeros(N); r = dict(dem=0, sold=0, lost=0, inv=0, so=0)
    for t in range(WARM, upto):
        sig = np.sqrt(np.mean(err[:, :t] ** 2, axis=1))                  # past forecast errors only
        order = np.maximum(0, np.ceil(np.maximum(Fc[:, t], 0) + z * sig - inv))
        avail = inv + order; d = A[:, t]; sold = np.minimum(avail, d); inv = avail - sold
        r["dem"] += d; r["sold"] += sold; r["lost"] += d - sold; r["inv"] += inv; r["so"] += d > avail
    r["n"] = upto - WARM; r["end"] = inv; return r
inv_rows, plan_oh = [], {}
for m, Fc in FC.items():
    for sl in SLS:
        r = simulate(Fc, norm.ppf(sl)); slp = 1 - r["so"] / r["n"]; av = r["inv"] / r["n"]
        for (s, c, ty), ix in grp.items():
            inv_rows.append([m, sl, s, c, ty, round(float(r["dem"][ix].sum()), 1), round(float(r["sold"][ix].sum()), 1), round(float(r["lost"][ix].sum()), 1),
                             round(float(av[ix].sum()), 2), round(float((av * price)[ix].sum()), 2), round(float((r["lost"] * price)[ix].sum()), 2),
                             round(float(slp[ix].sum()), 3), int((slp[ix] < 0.95).sum()), len(ix)])
        if m == "tso": plan_oh[sl] = simulate(Fc, norm.ppf(sl), upto=T - 1)["end"]   # stock on hand before the last order
inv_df = pd.DataFrame(inv_rows, columns=["m", "sl", "s", "c", "t", "dem", "sold", "lost", "inv", "val", "lval", "slsum", "below", "n"])
summ = inv_df[inv_df.sl == 0.95].groupby("m")[["dem", "sold", "inv", "slsum", "below", "n"]].sum()
summ["fill rate"] = summ.sold / summ.dem; summ["in-stock"] = summ.slsum / summ.n; summ["avg stock/product"] = summ.inv / summ.n
print("Inventory at 95% target:\n", summ[["fill rate", "in-stock", "avg stock/product", "below"]].round(4))
# ---- E. order plan for the last test day; F. top 20 products per store for the drill-down ----
sig_last = np.sqrt(np.mean((A - E)[:, :T - 1] ** 2, axis=1))
r4 = lambda x: round(float(x), 4)
plan = [[pp.store_id[i], pp.item_id[i], dept[i], dtype[i], r4(price[i]), r4(max(E[i, -1], 0)), r4(sig_last[i]), float(A[i, -1])] + [r4(plan_oh[sl][i]) for sl in SLS] for i in range(N)]
rev52 = REV[:, l52].sum(axis=1)
top_idx = [i for s in STORES for i in sorted(np.where(pp.store_id.values == s)[0], key=lambda i: -rev52[i])[:20]]
top = [{"s": pp.store_id[i], "i": pp.item_id[i], "a": A[i].tolist(), "f": np.round(E[i], 2).tolist(), "b": np.round(B[i], 2).tolist()} for i in top_idx]
DATA = {"meta": {"stores": STORES, "cats": ["FOODS", "HOUSEHOLD", "HOBBIES"], "types": ["smooth", "erratic", "intermittent", "lumpy"], "sls": SLS,
                 "test": [str(pd.Timestamp(dates[T0]).date()), str(pd.Timestamp(dates[T0 + T - 1]).date())], "warm": WARM, "n": N,
                 "planDate": str(pd.Timestamp(dates[T0 + T - 1]).date())},
        "weeks": week_start, "wk": wk_rows, "dept": dept_rows, "dates": [str(pd.Timestamp(d).date()) for d in dates[T0:T0 + T]],
        "fva": fva, "inv": inv_rows, "plan": plan, "top": top, "model": MODEL}
print(f"weeks {len(uw)} | products {N:,} | plan date {DATA['meta']['planDate']}")


Inventory at 95% target:
       fill rate  in-stock  avg stock/product  below
m                                                  
base     0.9581    0.9749             2.6590   1206
ma7      0.9571    0.9748             2.8195   1212
tso      0.9586    0.9752             2.6401   1202
weeks 72 | products 9,119 | plan date 2016-05-22


## Step 4 - Dashboard page (HTML + JavaScript)
This cell writes the dashboard template. `__DATA__` is replaced with the numbers computed above.

In [5]:
%%writefile dashboard_template.html
<title>TSO-DTF Decision Dashboard</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Archivo+Narrow:wght@500;600;700&family=Source+Sans+3:wght@400;600;700&family=IBM+Plex+Mono:wght@400;500&display=swap">
<style>
/* Layout: a store planner's workbench - tab strip for the four decisions, one sticky filter row, summary tiles, then chart and table panels that stack on phones */
:root {
  --bg: #f3f5f7; --panel: #ffffff; --ink: #141a22; --ink-2: #4d5766; --ink-3: #6f7886;
  --line: #dde2e8; --line-2: #eceff3; --accent: #2a78d6; --accent-ink: #1c5cab;
  --good: #0f7b52; --bad: #d03b3b; --warn: #a86a00; --mid: #eef0f2; --actual: #141a22;
  --s1: #2a78d6; --s2: #eb6834; --s3: #1baf7a; --s4: #eda100; --s5: #e87ba4; --s6: #7d8a99;
  --chip: #e7eef8; --band: rgba(42,120,214,.10);
  --f-display: "Archivo Narrow", "Arial Narrow", "Helvetica Neue", Arial, sans-serif;
  --f-body: "Source Sans 3", "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
  --f-mono: "IBM Plex Mono", ui-monospace, "SFMono-Regular", Menlo, Consolas, monospace;
}
@media (prefers-color-scheme: dark) { :root:not([data-theme="light"]) {
  --bg: #0f1216; --panel: #171b21; --ink: #eef1f5; --ink-2: #b3bbc6; --ink-3: #8c95a2;
  --line: #2a3139; --line-2: #22282f; --accent: #3987e5; --accent-ink: #86b6ef;
  --good: #3cc48d; --bad: #ec6b6b; --warn: #e0a53a; --mid: #262c33; --actual: #eef1f5;
  --s1: #3987e5; --s2: #d95926; --s3: #199e70; --s4: #c98500; --s5: #d55181; --s6: #95a1ae;
  --chip: #1f2a38; --band: rgba(57,135,229,.16); color-scheme: dark; } }
:root[data-theme="dark"] {
  --bg: #0f1216; --panel: #171b21; --ink: #eef1f5; --ink-2: #b3bbc6; --ink-3: #8c95a2;
  --line: #2a3139; --line-2: #22282f; --accent: #3987e5; --accent-ink: #86b6ef;
  --good: #3cc48d; --bad: #ec6b6b; --warn: #e0a53a; --mid: #262c33; --actual: #eef1f5;
  --s1: #3987e5; --s2: #d95926; --s3: #199e70; --s4: #c98500; --s5: #d55181; --s6: #95a1ae;
  --chip: #1f2a38; --band: rgba(57,135,229,.16); color-scheme: dark; }

* { box-sizing: border-box; }
[hidden] { display: none !important; }
body { background: var(--bg); color: var(--ink); font: 15px/1.5 var(--f-body); }
.wrap { max-width: 1240px; margin: 0 auto; padding-inline: 20px; padding-block: 24px 64px; display: grid; gap: 18px; }
h1, h2, h3 { font-family: var(--f-display); font-weight: 600; margin: 0; text-wrap: balance; }
h1 { font-size: clamp(28px, 4.2vw, 40px); line-height: 1.05; font-weight: 700; }
h2 { font-size: 20px; line-height: 1.2; }
p { margin: 0; }
.eyebrow { font: 600 12px/1.3 var(--f-body); text-transform: uppercase; letter-spacing: 0.09em; color: var(--ink-3); }
.mono { font-family: var(--f-mono); font-variant-numeric: tabular-nums; }
.muted { color: var(--ink-2); }
.small { font-size: 13px; }
header.top { display: grid; gap: 8px; }
header.top .lede { max-width: 76ch; color: var(--ink-2); }

.tabs { display: flex; gap: 4px; border-bottom: 1px solid var(--line); overflow-x: auto; }
.tabs button { font: 600 15px/1 var(--f-display); letter-spacing: .01em; color: var(--ink-2); background: none; border: 0; border-bottom: 3px solid transparent;
  padding: 12px 14px 10px; cursor: pointer; white-space: nowrap; }
.tabs button[aria-selected="true"] { color: var(--ink); border-bottom-color: var(--accent); }
.tabs button:focus-visible, .seg button:focus-visible, select:focus-visible, input:focus-visible, .more:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }

.filters { position: sticky; top: env(safe-area-inset-top, 0px); z-index: 5; background: var(--bg); border-bottom: 1px solid var(--line); padding-block: 10px; display: flex; flex-wrap: wrap; gap: 10px 18px; align-items: center; }
.fgroup { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
.fgroup > span { font: 600 12px/1 var(--f-body); text-transform: uppercase; letter-spacing: 0.08em; color: var(--ink-3); }
.seg { display: inline-flex; border: 1px solid var(--line); border-radius: 8px; overflow: hidden; background: var(--panel); flex-wrap: wrap; }
.seg button { font: 600 13px/1 var(--f-body); color: var(--ink-2); background: transparent; border: 0; padding: 8px 11px; cursor: pointer; border-right: 1px solid var(--line); }
.seg button:last-child { border-right: 0; }
.seg button[aria-pressed="true"] { background: var(--accent); color: #fff; }
.count { margin-left: auto; font: 500 13px/1 var(--f-mono); color: var(--ink-2); }

.tiles { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 12px; }
.tile { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 14px 16px; display: grid; gap: 4px; align-content: start; min-width: 0; }
.tile .big { font: 600 27px/1.1 var(--f-mono); font-variant-numeric: tabular-nums; }
.tile .lbl { font-size: 13px; color: var(--ink-2); }
.tile .sub { font: 500 12.5px/1.3 var(--f-mono); }
.good { color: var(--good); } .bad { color: var(--bad); } .warn { color: var(--warn); }

.grid2 { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 18px; }
.panel { background: var(--panel); border: 1px solid var(--line); border-radius: 12px; padding: 18px; display: grid; gap: 12px; align-content: start; min-width: 0; }
.panel.span2 { grid-column: 1 / -1; }
.panel .head { display: grid; gap: 4px; }
.panel .head.row { display: flex; justify-content: space-between; align-items: end; flex-wrap: wrap; gap: 10px; }
.chart { width: 100%; min-width: 0; position: relative; }
.chart svg { display: block; width: 100%; height: auto; overflow: visible; }
.chart text { font-family: var(--f-body); fill: var(--ink-2); font-size: 12px; }
.chart .t-ink { fill: var(--ink); }
.chart .t-num { font-family: var(--f-mono); font-variant-numeric: tabular-nums; }
.legend { display: flex; flex-wrap: wrap; gap: 6px 14px; font-size: 13px; color: var(--ink-2); }
.legend i { display: inline-block; width: 10px; height: 10px; border-radius: 2px; margin-right: 6px; vertical-align: -1px; }
.legend i.line { height: 2px; width: 16px; border-radius: 1px; vertical-align: 3px; }
.legend i.dash { height: 0; width: 16px; border-top: 2px dashed; border-radius: 0; vertical-align: 3px; background: none !important; }

.tablewrap { overflow-x: auto; }
table { border-collapse: collapse; width: 100%; font-size: 13.5px; }
th, td { padding: 7px 10px; text-align: right; border-bottom: 1px solid var(--line-2); white-space: nowrap; }
th:first-child, td:first-child { text-align: left; }
thead th { font: 600 12px/1.3 var(--f-body); text-transform: uppercase; letter-spacing: 0.06em; color: var(--ink-3); border-bottom: 1px solid var(--line); vertical-align: bottom; }
thead th.sortable { cursor: pointer; }
thead th.sortable:hover { color: var(--ink); }
td.num { font-family: var(--f-mono); font-variant-numeric: tabular-nums; }
tr.hl td { font-weight: 700; background: color-mix(in oklab, var(--accent) 8%, var(--panel)); }
.pill { display: inline-block; font: 600 11px/1 var(--f-body); padding: 4px 7px; border-radius: 999px; }
.pill.smooth { background: color-mix(in oklab, var(--s1) 18%, var(--panel)); color: var(--ink); }
.pill.erratic { background: color-mix(in oklab, var(--s2) 20%, var(--panel)); color: var(--ink); }
.pill.intermittent { background: color-mix(in oklab, var(--s3) 20%, var(--panel)); color: var(--ink); }
.pill.lumpy { background: color-mix(in oklab, var(--s5) 22%, var(--panel)); color: var(--ink); }
.pill.order { background: color-mix(in oklab, var(--accent) 18%, var(--panel)); color: var(--accent-ink); }
.pill.none { background: var(--mid); color: var(--ink-2); }

select, input[type="search"] { font: 600 13px/1 var(--f-body); color: var(--ink); background: var(--panel); border: 1px solid var(--line); border-radius: 8px; padding: 8px 10px; max-width: 100%; }
input[type="search"] { font-weight: 400; min-width: 0; width: 220px; }
.row { display: flex; flex-wrap: wrap; gap: 10px 16px; align-items: center; }
.kv { display: flex; gap: 6px; align-items: baseline; font-size: 13px; color: var(--ink-2); }
.kv b { font: 600 14px var(--f-mono); color: var(--ink); }
.callout { border-left: 3px solid var(--accent); padding: 10px 14px; background: color-mix(in oklab, var(--accent) 7%, var(--panel)); border-radius: 0 8px 8px 0; font-size: 14.5px; }
.callout b { font-family: var(--f-mono); }
.notes { display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 12px 24px; font-size: 14px; color: var(--ink-2); }
.notes b { color: var(--ink); }
.notes div { display: grid; gap: 4px; }
.more { font: 600 13px/1 var(--f-body); color: var(--accent-ink); background: none; border: 1px solid var(--line); border-radius: 8px; padding: 8px 12px; cursor: pointer; justify-self: start; }
#tip { position: fixed; pointer-events: none; z-index: 20; background: var(--panel); color: var(--ink); border: 1px solid var(--line);
  border-radius: 8px; padding: 8px 10px; font-size: 12.5px; line-height: 1.45; box-shadow: 0 6px 20px rgba(0,0,0,.14); max-width: 290px; }
#tip .n { font-family: var(--f-mono); font-variant-numeric: tabular-nums; }
@media (max-width: 900px) { .grid2 { grid-template-columns: minmax(0, 1fr); } }
@media (max-width: 480px) { .wrap { padding-inline: 16px; } .tile .big { font-size: 22px; } .count { margin-left: 0; } input[type="search"] { width: 100%; } }
@media (prefers-reduced-motion: reduce) { * { transition: none !important; } }
</style>

<div class="wrap">
  <header class="top">
    <div class="eyebrow">Decision Tree Optimization Framework for Advanced Decision Support and Business Analytics · Walmart M5 · CA_1, TX_1, WI_1</div>
    <h1>TSO-DTF Decision Dashboard</h1>
    <p class="lede">Turns the TSO-DTF forecasts into business decisions and compares them with the base paper's DTF: what the stores sell, what they will sell tomorrow, how much stock to hold, how much to order, and how accurate each model is. Pick a store, category or demand type and every view updates.</p>
  </header>

  <nav class="tabs" role="tablist" aria-label="Views">
    <button role="tab" id="tab-sales" aria-controls="v-sales" aria-selected="true">Sales overview</button>
    <button role="tab" id="tab-fc" aria-controls="v-fc" aria-selected="false">Demand forecast</button>
    <button role="tab" id="tab-inv" aria-controls="v-inv" aria-selected="false">Inventory planning</button>
    <button role="tab" id="tab-order" aria-controls="v-order" aria-selected="false">Order plan</button>
    <button role="tab" id="tab-model" aria-controls="v-model" aria-selected="false">Model accuracy</button>
  </nav>

  <div class="filters" role="region" aria-label="Filters">
    <div class="fgroup"><span>Store</span><div class="seg" id="f-store"></div></div>
    <div class="fgroup"><span>Category</span><div class="seg" id="f-cat"></div></div>
    <div class="fgroup"><span>Demand type</span><div class="seg" id="f-type"></div></div>
    <div class="count" id="count"></div>
  </div>

  <!-- ============ SALES ============ -->
  <section id="v-sales" role="tabpanel" aria-labelledby="tab-sales" style="display:grid;gap:18px">
    <div class="tiles" id="sales-tiles"></div>
    <div class="panel span2">
      <div class="head row">
        <div class="head"><h2>Weekly sales</h2><p class="muted small" id="trend-sub"></p></div>
        <div class="seg" id="f-measure"><button type="button" id="m-rev" data-v="r" aria-pressed="true">Revenue ($)</button><button type="button" id="m-units" data-v="u" aria-pressed="false">Units</button></div>
      </div>
      <div class="chart" id="c-trend"></div>
      <div class="legend" id="trend-legend"></div>
    </div>
    <div class="grid2">
      <div class="panel">
        <div class="head"><h2>Revenue by department</h2><p class="muted small">Last 52 weeks before 22 Mar 2016.</p></div>
        <div class="chart" id="c-dept"></div>
      </div>
      <div class="panel">
        <div class="head"><h2>Demand types</h2><p class="muted small">How regularly products sell (Syntetos ADI/CV² classes). Most products sell on some days only, which makes them hard to stock.</p></div>
        <div class="chart" id="c-types"></div>
      </div>
    </div>
  </section>

  <!-- ============ FORECAST ============ -->
  <section id="v-fc" role="tabpanel" aria-labelledby="tab-fc" hidden style="display:grid;gap:18px">
    <div class="tiles" id="fc-tiles"></div>
    <div class="panel span2">
      <div class="head"><h2>Daily units: forecast vs actual</h2><p class="muted small" id="fc-sub"></p></div>
      <div class="chart" id="c-fc"></div>
      <div class="legend"><span><i class="line" style="background:var(--actual)"></i>Actual units sold</span><span><i class="line" style="background:var(--s1)"></i>TSO-DTF forecast</span><span><i class="dash" style="border-color:var(--s2)"></i>Base paper DTF</span></div>
    </div>
    <div class="panel span2">
      <div class="head row">
        <div class="head"><h2>Product drill-down</h2><p class="muted small">Top 20 products by revenue in each store.</p></div>
        <select id="f-prod" aria-label="Product"></select>
      </div>
      <div class="row" id="prod-kv"></div>
      <div class="chart" id="c-prod"></div>
    </div>
  </section>

  <!-- ============ INVENTORY ============ -->
  <section id="v-inv" role="tabpanel" aria-labelledby="tab-inv" hidden style="display:grid;gap:18px">
    <div class="panel span2">
      <div class="head row">
        <div class="head"><h2>Service target</h2><p class="muted small">Chance that a product does not run out on a given day. Higher targets need more safety stock.</p></div>
        <div class="seg" id="f-sl"></div>
      </div>
      <div class="callout" id="inv-callout"></div>
    </div>
    <div class="tiles" id="inv-tiles"></div>
    <div class="grid2">
      <div class="panel">
        <div class="head"><h2>Stock needed for each fill rate</h2><p class="muted small">Each point is one service target. Lower and further right is better: more demand served with less stock.</p></div>
        <div class="chart" id="c-trade"></div>
        <div class="legend" id="trade-legend"></div>
      </div>
      <div class="panel">
        <div class="head"><h2>Ordering rules compared</h2><p class="muted small" id="cmp-sub"></p></div>
        <div class="tablewrap"><table id="t-cmp"></table></div>
        <p class="muted small">Base paper DTF is the model this project improves on; the 7-day average is a simple rule many stores use without a model.</p>
      </div>
    </div>
    <div class="panel span2">
      <div class="head"><h2>By category</h2><p class="muted small">TSO-DTF policy at the chosen target.</p></div>
      <div class="tablewrap"><table id="t-invcat"></table></div>
    </div>
  </section>

  <!-- ============ ORDER PLAN ============ -->
  <section id="v-order" role="tabpanel" aria-labelledby="tab-order" hidden style="display:grid;gap:18px">
    <div class="tiles" id="order-tiles"></div>
    <div class="panel span2">
      <div class="head row">
        <div class="head"><h2 id="order-title">Order plan</h2><p class="muted small">Order-up-to level = tomorrow's forecast + safety stock. Order = order-up-to level − stock on hand, rounded up to whole units.</p></div>
        <div class="row"><div class="seg" id="f-sl2"></div><input type="search" id="f-search" placeholder="Search product, e.g. FOODS_3_090" aria-label="Search product"></div>
      </div>
      <div class="tablewrap"><table id="t-order"></table></div>
      <button type="button" class="more" id="order-more">Show 25 more</button>
    </div>
  </section>

  <!-- ============ MODEL ============ -->
  <section id="v-model" role="tabpanel" aria-labelledby="tab-model" hidden style="display:grid;gap:18px">
    <div class="tiles" id="model-tiles"></div>
    <div class="grid2">
      <div class="panel">
        <div class="head"><h2>Improvement at each M5 level</h2><p class="muted small">WRMSSE change of TSO-DTF vs the base paper DTF. Higher is better.</p></div>
        <div class="chart" id="c-levels"></div>
      </div>
      <div class="panel">
        <div class="head"><h2>Accuracy by level (WRMSSE)</h2><p class="muted small">Official M5 metric; lower is better. The official score is the mean of the 12 M5 levels.</p></div>
        <div class="tablewrap"><table id="t-levels"></table></div>
      </div>
    </div>
    <div class="grid2">
      <div class="panel">
        <div class="head"><h2>Product-level metrics</h2><p class="muted small">All 9,119 product-store series, 62 test days. Lower is better (SME: closer to 0).</p></div>
        <div class="tablewrap"><table id="t-metrics"></table></div>
      </div>
      <div class="panel">
        <div class="head"><h2>Ablation: each optimization step</h2><p class="muted small">Change vs the base paper DTF. The final model averages Opt 4, 5, 6, 7 and 10.</p></div>
        <div class="tablewrap"><table id="t-model"></table></div>
      </div>
    </div>
    <div class="panel span2">
      <div class="head"><h2>How this planner works</h2></div>
      <div class="notes">
        <div><b>Forecast</b><span>TSO-DTF final forecast: equal-weight average of the Opt 4, 5, 6, 7 and 10 LightGBM models. It predicts each product's sales for the next day, using sales up to today.</span></div>
        <div><b>Stock policy</b><span>Every evening the store orders up to tomorrow's forecast plus safety stock; the order arrives next morning. Unmet demand is lost.</span></div>
        <div><b>Safety stock</b><span>z × the product's forecast error (RMSE) measured on earlier days only, where z comes from the service target (95% → 1.645).</span></div>
        <div><b>Test period</b><span>22 Mar – 22 May 2016. The first 14 days only measure forecast errors; stock results cover 5 Apr – 22 May (48 days).</span></div>
        <div><b>Stock value</b><span>Average units on the shelf at the end of each day × the product's latest selling price.</span></div>
        <div><b>Base paper DTF</b><span>The base paper's LightGBM per store with fixed features and Optuna TPE (21 trials), re-run on the same data and days.</span></div>
        <div><b>Link to base paper</b><span>Section 6 of the base paper also turns forecasts into an inventory policy and reports mean inventory, service level and products below 95%.</span></div>
      </div>
    </div>
  </section>
</div>
<div id="tip" hidden></div>

<script id="data" type="application/json">__DATA__</script>
<script>
(() => {
  const D = JSON.parse(document.getElementById("data").textContent);
  const M = D.meta, STORES = M.stores, CATS = M.cats, TYPES = M.types, SLS = M.sls;
  const TYPE_LABEL = { smooth: "Smooth", erratic: "Erratic", intermittent: "Intermittent", lumpy: "Lumpy" };
  const CAT_LABEL = { FOODS: "Foods", HOUSEHOLD: "Household", HOBBIES: "Hobbies" };
  const POL = { tso: "TSO-DTF forecast", base: "Base paper DTF", ma7: "7-day average" };
  const css = v => getComputedStyle(document.documentElement).getPropertyValue(v).trim();
  const state = { store: -1, cat: -1, type: -1, tab: "sales", measure: "r", sl: 3, prod: 0, sortKey: "val", sortDir: -1, shown: 25, q: "" };
  try { const s = JSON.parse(localStorage.getItem("planner-state") || "{}"); for (const k of ["store","cat","type","tab","measure","sl"]) if (k in s) state[k] = s[k]; } catch (e) {}
  const save = () => { try { localStorage.setItem("planner-state", JSON.stringify({ store: state.store, cat: state.cat, type: state.type, tab: state.tab, measure: state.measure, sl: state.sl })); } catch (e) {} };
  const hash = location.hash.replace("#", ""); if (["sales","fc","inv","order","model"].includes(hash)) state.tab = hash;

  const keep = (s, c, t) => (state.store < 0 || s === STORES[state.store]) && (state.cat < 0 || c === CATS[state.cat]) && (state.type < 0 || t === TYPES[state.type]);
  const fmtInt = v => Math.round(v).toLocaleString("en-US");
  const fmtMoney = v => (Math.abs(v) >= 1e6 ? "$" + (v / 1e6).toFixed(2) + "M" : Math.abs(v) >= 1e4 ? "$" + (v / 1e3).toFixed(1) + "k" : "$" + Math.round(v).toLocaleString("en-US"));
  const pct = (v, d = 1) => (100 * v).toFixed(d) + "%";
  const sgn = (v, d = 1) => (v >= 0 ? "+" : "−") + Math.abs(v).toFixed(d) + "%";
  const esc = s => String(s).replace(/[&<>"]/g, c => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
  const fmtDate = s => { const d = new Date(s + "T00:00:00"); return d.toLocaleDateString("en-GB", { day: "numeric", month: "short" }); };
  const tip = document.getElementById("tip");
  const showTip = (html, ev) => { tip.innerHTML = html; tip.hidden = false; const w = tip.offsetWidth, h = tip.offsetHeight;
    let x = ev.clientX + 14, y = ev.clientY + 14; if (x + w > innerWidth - 8) x = ev.clientX - w - 14; if (y + h > innerHeight - 8) y = ev.clientY - h - 14; tip.style.left = Math.max(8, x) + "px"; tip.style.top = Math.max(8, y) + "px"; };
  const hideTip = () => { tip.hidden = true; };
  const nice = (max, n = 4) => { if (max <= 0) return [1, [0, 1]]; const raw = max / n, p = Math.pow(10, Math.floor(Math.log10(raw))), f = raw / p; const step = (f <= 1 ? 1 : f <= 2 ? 2 : f <= 2.5 ? 2.5 : f <= 5 ? 5 : 10) * p; const top = Math.ceil(max / step) * step; const t = []; for (let v = 0; v <= top + 1e-9; v += step) t.push(v); return [top, t]; };
  const short = v => v >= 1e6 ? (v / 1e6).toFixed(v >= 1e7 ? 0 : 1) + "M" : v >= 1e3 ? (v / 1e3).toFixed(v >= 1e4 ? 0 : 1) + "k" : String(Math.round(v * 100) / 100);

  // ---------- filters & tabs ----------
  function seg(el, items, key, cb) {
    el.innerHTML = items.map(([v, l]) => `<button type="button" id="${el.id}-${String(v).replace(".", "_")}" data-v="${v}" aria-pressed="${state[key] === v}">${l}</button>`).join("");
    el.addEventListener("click", e => { const b = e.target.closest("button"); if (!b) return; state[key] = +b.dataset.v; state.shown = 25; save(); (cb || render)(); });
  }
  const segSync = (id, key) => document.querySelectorAll(`#${id} button`).forEach(b => b.setAttribute("aria-pressed", String(+b.dataset.v === state[key])));
  seg(document.getElementById("f-store"), [[-1, "All"], ...STORES.map((s, i) => [i, s])], "store");
  seg(document.getElementById("f-cat"), [[-1, "All"], ...CATS.map((c, i) => [i, CAT_LABEL[c]])], "cat");
  seg(document.getElementById("f-type"), [[-1, "All"], ...TYPES.map((t, i) => [i, TYPE_LABEL[t]])], "type");
  const slItems = SLS.map((s, i) => [i, (s * 100) % 1 ? (s * 100).toFixed(1) + "%" : (s * 100) + "%"]);
  seg(document.getElementById("f-sl"), slItems, "sl");
  seg(document.getElementById("f-sl2"), slItems, "sl");
  document.getElementById("f-measure").addEventListener("click", e => { const b = e.target.closest("button"); if (!b) return; state.measure = b.dataset.v; save(); render(); });
  document.querySelectorAll(".tabs button").forEach(b => b.addEventListener("click", () => { state.tab = b.id.replace("tab-", ""); save(); render(); }));
  document.getElementById("f-search").addEventListener("input", e => { state.q = e.target.value.trim().toUpperCase(); state.shown = 25; renderOrder(); });
  document.getElementById("order-more").addEventListener("click", () => { state.shown += 25; renderOrder(); });

  // ---------- generic line chart ----------
  function lineChart(el, xs, series, opts = {}) {
    const W = Math.max(300, el.clientWidth), H = opts.h || (W < 560 ? 230 : 290), pad = { l: 52, r: 12, t: 12, b: 28 };
    const max = Math.max(...series.flatMap(s => s.v.filter(v => v != null)), 0);
    const [top, ticks] = nice(max * 1.05);
    const sx = i => pad.l + (W - pad.l - pad.r) * (xs.length === 1 ? 0.5 : i / (xs.length - 1)), sy = v => pad.t + (H - pad.t - pad.b) * (1 - v / top);
    let s = ticks.map(t => `<line x1="${pad.l}" x2="${W - pad.r}" y1="${sy(t)}" y2="${sy(t)}" stroke="${css("--line-2")}"/><text x="${pad.l - 8}" y="${sy(t) + 4}" text-anchor="end" class="t-num">${opts.money ? "$" + short(t) : short(t)}</text>`).join("");
    const nLab = W < 560 ? 4 : 7, step = Math.max(1, Math.round((xs.length - 1) / (nLab - 1)));
    for (let i = 0; i < xs.length; i += step) s += `<text x="${sx(i)}" y="${H - 8}" text-anchor="middle">${opts.xfmt ? opts.xfmt(xs[i]) : xs[i]}</text>`;
    if (opts.band) s += `<rect x="${sx(opts.band[0])}" y="${pad.t}" width="${sx(opts.band[1]) - sx(opts.band[0])}" height="${H - pad.t - pad.b}" fill="${css("--band")}"/><text x="${sx(opts.band[0]) + 6}" y="${pad.t + 14}" style="font-size:11px">${opts.bandLabel || ""}</text>`;
    for (const se of series) {
      const pts = se.v.map((v, i) => v == null ? null : `${sx(i).toFixed(1)},${sy(v).toFixed(1)}`).filter(Boolean).join(" ");
      if (se.area) s += `<polygon points="${sx(0)},${sy(0)} ${pts} ${sx(se.v.length - 1)},${sy(0)}" fill="${se.c}" opacity="0.10"/>`;
      s += `<polyline points="${pts}" fill="none" stroke="${se.c}" stroke-width="${se.w || 2}" ${se.dash ? 'stroke-dasharray="5 4"' : ""} stroke-linejoin="round" stroke-linecap="round"/>`;
    }
    s += `<line id="${el.id}-x" x1="0" x2="0" y1="${pad.t}" y2="${H - pad.b}" stroke="${css("--ink-3")}" stroke-dasharray="3 3" visibility="hidden"/>`;
    el.innerHTML = `<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="${esc(opts.label || "chart")}">${s}<rect x="${pad.l}" y="${pad.t}" width="${W - pad.l - pad.r}" height="${H - pad.t - pad.b}" fill="transparent" class="hov"/></svg>`;
    const svg = el.querySelector("svg"), ln = svg.querySelector(`#${el.id}-x`);
    svg.querySelector(".hov").addEventListener("mousemove", ev => {
      const r = svg.getBoundingClientRect(), x = (ev.clientX - r.left) * W / r.width;
      const i = Math.max(0, Math.min(xs.length - 1, Math.round((x - pad.l) / (W - pad.l - pad.r) * (xs.length - 1))));
      ln.setAttribute("x1", sx(i)); ln.setAttribute("x2", sx(i)); ln.setAttribute("visibility", "visible");
      showTip(`<b>${opts.xfmt ? opts.xfmt(xs[i]) : xs[i]}${opts.xsuffix || ""}</b><br>` + series.map(se => `<span style="color:${se.c}">■</span> ${se.name}: <span class="n">${se.v[i] == null ? "–" : (opts.money ? fmtMoney(se.v[i]) : fmtInt(se.v[i]))}</span>`).join("<br>"), ev);
    });
    svg.querySelector(".hov").addEventListener("mouseleave", () => { ln.setAttribute("visibility", "hidden"); hideTip(); });
  }
  function hbars(el, rows, opts = {}) {
    const W = Math.max(300, el.clientWidth), lw = opts.lw || 110, rh = 28, H = rows.length * rh + 8;
    const max = Math.max(...rows.map(r => r.v), 1), x1 = W - (opts.rw || 70);
    let s = "";
    rows.forEach((r, i) => {
      const y = i * rh + 4, w = Math.max(1, (x1 - lw) * r.v / max);
      s += `<g class="hb" data-i="${i}"><rect x="0" y="${y}" width="${W}" height="${rh}" fill="transparent"/>
        <text x="0" y="${y + rh / 2 + 4}" class="t-ink">${esc(r.label)}</text>
        <rect x="${lw}" y="${y + 6}" width="${w}" height="${rh - 12}" rx="3" fill="${r.c || css("--s1")}"/>
        <text x="${lw + w + 6}" y="${y + rh / 2 + 4}" class="t-num">${r.txt}</text></g>`;
    });
    el.innerHTML = `<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="${esc(opts.label || "bar chart")}">${s}</svg>`;
    el.querySelectorAll(".hb").forEach(g => { const r = rows[+g.dataset.i]; if (!r.tip) return; g.addEventListener("mousemove", ev => showTip(r.tip, ev)); g.addEventListener("mouseleave", hideTip); });
  }

  // ---------- SALES ----------
  const NW = D.weeks.length;
  function renderSales() {
    const rows = D.wk.filter(r => keep(r[1], r[2], r[3]));
    const byStore = state.store < 0;
    const groups = byStore ? STORES : CATS, gi = byStore ? 1 : 2, col = [css("--s1"), css("--s2"), css("--s3")];
    const mi = state.measure === "r" ? 5 : 4;
    const ser = groups.map((g, k) => { const v = new Array(NW).fill(0); rows.forEach(r => { if (r[gi] === g) v[r[0]] += r[mi]; }); return { name: byStore ? g : CAT_LABEL[g], v, c: col[k] }; }).filter(s => s.v.some(x => x > 0));
    lineChart(document.getElementById("c-trend"), D.weeks, ser, { money: state.measure === "r", xfmt: s => { const d = new Date(s + "T00:00:00"); return d.toLocaleDateString("en-GB", { month: "short", year: "2-digit" }); }, xsuffix: "", label: "Weekly sales", band: [NW - 9, NW - 1], bandLabel: "Test period" });
    document.getElementById("trend-legend").innerHTML = ser.map(s => `<span><i class="line" style="background:${s.c}"></i>${s.name}</span>`).join("");
    document.getElementById("trend-sub").textContent = `${state.measure === "r" ? "Revenue" : "Units sold"} per Walmart week (Saturday to Friday), Jan 2015 – May 2016, ${byStore ? "one line per store" : "one line per category"}. Shaded: the forecast test period.`;
    segSync("f-measure", "measure"); document.querySelectorAll("#f-measure button").forEach(b => b.setAttribute("aria-pressed", String(b.dataset.v === state.measure)));
    // tiles: last 52 weeks; last 13 weeks vs same weeks a year earlier
    const sum = (a, b, i) => rows.filter(r => r[0] >= a && r[0] <= b).reduce((t, r) => t + r[i], 0);
    const last = NW - 1, r52 = sum(last - 51, last, 5), u52 = sum(last - 51, last, 4), r13 = sum(last - 12, last, 5), r13p = sum(last - 12 - 52, last - 52, 5);
    const nProd = D.plan.filter(p => keep(p[0], p[1].split("_")[0], p[3])).length, g = (r13 - r13p) / r13p;
    document.getElementById("sales-tiles").innerHTML = [
      [fmtMoney(r52), "Revenue, last 52 weeks", `${fmtMoney(r52 / 52)} per week`],
      [fmtInt(u52), "Units sold, last 52 weeks", `${fmtInt(u52 / 364)} per day`],
      [sgn(100 * g), "Revenue, last 13 weeks vs a year earlier", `${fmtMoney(r13)} vs ${fmtMoney(r13p)}`, g >= 0 ? "good" : "bad"],
      [fmtInt(nProd), "Products in view", `${fmtMoney(r52 / Math.max(1, nProd))} revenue per product`]
    ].map(([b, l, s, c]) => `<div class="tile"><div class="lbl">${l}</div><div class="big ${c || ""}">${b}</div><div class="sub muted">${s}</div></div>`).join("");
    // departments
    const dep = {}; D.dept.filter(r => keep(r[1], r[0].split("_")[0], r[2])).forEach(r => { dep[r[0]] = (dep[r[0]] || 0) + r[4]; });
    const catCol = { FOODS: css("--s1"), HOUSEHOLD: css("--s2"), HOBBIES: css("--s3") };
    const tot = Object.values(dep).reduce((a, b) => a + b, 0) || 1;
    hbars(document.getElementById("c-dept"), Object.entries(dep).sort((a, b) => b[1] - a[1]).map(([k, v]) => ({ label: k, v, c: catCol[k.split("_")[0]], txt: `${fmtMoney(v)} · ${pct(v / tot, 0)}`, tip: `<b>${k}</b><br>Revenue <span class="n">${fmtMoney(v)}</span><br>Share <span class="n">${pct(v / tot)}</span>` })), { lw: 96, rw: 120, label: "Revenue by department" });
    // demand types
    const tc = {}, tr = {}; D.plan.filter(p => keep(p[0], p[1].split("_")[0], p[3])).forEach(p => { tc[p[3]] = (tc[p[3]] || 0) + 1; });
    D.dept.filter(r => keep(r[1], r[0].split("_")[0], r[2])).forEach(r => { tr[r[2]] = (tr[r[2]] || 0) + r[4]; });
    const tcol = { smooth: css("--s1"), erratic: css("--s2"), intermittent: css("--s3"), lumpy: css("--s5") }, np = Object.values(tc).reduce((a, b) => a + b, 0) || 1;
    hbars(document.getElementById("c-types"), TYPES.filter(t => tc[t]).map(t => ({ label: TYPE_LABEL[t], v: tc[t], c: tcol[t], txt: `${fmtInt(tc[t])} products · ${pct((tr[t] || 0) / tot, 0)} revenue`, tip: `<b>${TYPE_LABEL[t]}</b><br>${fmtInt(tc[t])} products (${pct(tc[t] / np)})<br>Revenue share <span class="n">${pct((tr[t] || 0) / tot)}</span>` })), { lw: 96, rw: 250, label: "Products by demand type" });
  }

  // ---------- FORECAST ----------
  function renderFc() {
    const T = D.dates.length, a = new Array(T).fill(0), f = new Array(T).fill(0), n = new Array(T).fill(0), m = new Array(T).fill(0);
    D.fva.filter(r => keep(r[1], r[2], r[3])).forEach(r => { a[r[0]] += r[4]; f[r[0]] += r[5]; n[r[0]] += r[6]; m[r[0]] += r[7]; });   // n = base paper DTF
    lineChart(document.getElementById("c-fc"), D.dates, [{ name: "Actual", v: a, c: css("--actual"), w: 1.6 }, { name: "TSO-DTF forecast", v: f, c: css("--s1"), w: 2.2, area: true }, { name: "Base paper DTF", v: n, c: css("--s2"), w: 1.5, dash: true }], { xfmt: fmtDate, label: "Daily forecast vs actual" });
    const acc = x => 1 - x.reduce((t, v, i) => t + Math.abs(v - a[i]), 0) / a.reduce((t, v) => t + v, 0);
    const bias = x => x.reduce((t, v, i) => t + v - a[i], 0) / a.reduce((t, v) => t + v, 0);
    const at = acc(f), am = acc(m), an = acc(n), sa = a.reduce((t, v) => t + v, 0);
    document.getElementById("fc-tiles").innerHTML = [
      [pct(at), "TSO-DTF forecast accuracy", "1 − WAPE on daily totals in view", "good"],
      [pct(an), "Base paper DTF accuracy", `${sgn(100 * (at - an), 2)} points for TSO-DTF`],
      [pct(am), "7-day average accuracy", `${sgn(100 * (at - am))} points for TSO-DTF`],
      [sgn(100 * bias(f)), "TSO-DTF bias", bias(f) < 0 ? "forecasts slightly below sales" : "forecasts slightly above sales"],
      [fmtInt(sa), "Units sold in test period", `${fmtInt(sa / T)} per day`]
    ].map(([b, l, s, c]) => `<div class="tile"><div class="lbl">${l}</div><div class="big ${c || ""}">${b}</div><div class="sub muted">${s}</div></div>`).join("");
    document.getElementById("fc-sub").textContent = `Total daily units for the selection, 22 Mar – 22 May 2016. Each forecast is made the evening before with the sales known up to then.`;
    // product drill-down
    const sel = document.getElementById("f-prod");
    const opts = D.top.map((p, i) => [i, p]).filter(([i, p]) => keep(p.s, p.i.split("_")[0], (D.plan.find(q => q[0] === p.s && q[1] === p.i) || [])[3]));
    const cur = opts.some(([i]) => i === state.prod) ? state.prod : (opts[0] || [0])[0]; state.prod = cur;
    sel.innerHTML = opts.length ? opts.map(([i, p]) => `<option value="${i}" ${i === cur ? "selected" : ""}>${p.s} · ${p.i}</option>`).join("") : `<option>No top product in this selection</option>`;
    const P = D.top[cur];
    if (!opts.length) { document.getElementById("c-prod").innerHTML = `<p class="muted small">None of the top-selling products match these filters. Choose another store, category or demand type.</p>`; document.getElementById("prod-kv").innerHTML = ""; return; }
    const pa = P.a, err = (x) => { let e = 0, s = 0; x.forEach((v, i) => { e += Math.abs(v - pa[i]); s += pa[i]; }); return 1 - e / s; };
    const info = D.plan.find(q => q[0] === P.s && q[1] === P.i);
    document.getElementById("prod-kv").innerHTML = `<span class="kv">Price <b>$${info[4].toFixed(2)}</b></span><span class="kv">Demand type <span class="pill ${info[3]}">${TYPE_LABEL[info[3]]}</span></span><span class="kv">TSO-DTF accuracy <b>${pct(err(P.f))}</b></span><span class="kv">Base paper DTF accuracy <b>${pct(err(P.b))}</b></span>`;
    lineChart(document.getElementById("c-prod"), D.dates, [{ name: "Actual", v: pa, c: css("--actual"), w: 1.5 }, { name: "TSO-DTF forecast", v: P.f, c: css("--s1"), w: 2.2 }, { name: "Base paper DTF", v: P.b, c: css("--s2"), w: 1.4, dash: true }], { xfmt: fmtDate, h: 240, label: "Product forecast vs actual" });
  }
  document.getElementById("f-prod").addEventListener("change", e => { state.prod = +e.target.value; renderFc(); });

  // ---------- INVENTORY ----------
  function invAgg(m, sl) {
    const t = { dem: 0, sold: 0, lost: 0, inv: 0, val: 0, lval: 0, slsum: 0, below: 0, n: 0 };
    D.inv.forEach(r => { if (r[0] !== m || r[1] !== sl || !keep(r[2], r[3], r[4])) return; t.dem += r[5]; t.sold += r[6]; t.lost += r[7]; t.inv += r[8]; t.val += r[9]; t.lval += r[10]; t.slsum += r[11]; t.below += r[12]; t.n += r[13]; });
    t.fill = t.sold / (t.dem || 1); t.svc = t.slsum / (t.n || 1); return t;
  }
  // stock value another rule needs to reach the same fill rate (linear interpolation along its curve)
  function valAtFill(m, fill) {
    const c = SLS.map(s => invAgg(m, s)).sort((a, b) => a.fill - b.fill);
    if (fill <= c[0].fill) return null; if (fill > c[c.length - 1].fill) return null;
    for (let i = 1; i < c.length; i++) if (fill <= c[i].fill) { const w = (fill - c[i - 1].fill) / (c[i].fill - c[i - 1].fill); return c[i - 1].val + w * (c[i].val - c[i - 1].val); }
    return null;
  }
  function renderInv() {
    const sl = SLS[state.sl], T = invAgg("tso", sl), NV = invAgg("base", sl), MA = invAgg("ma7", sl);
    const vMA = valAtFill("ma7", T.fill), vNV = valAtFill("base", T.fill);
    let call = `At a <b>${pct(sl, sl * 100 % 1 ? 1 : 0)}</b> service target, ordering with TSO-DTF forecasts serves <b>${pct(T.fill)}</b> of demand while holding <b>${fmtMoney(T.val)}</b> of stock on average.`;
    call += ` With the base paper DTF the same rule serves <b>${pct(NV.fill)}</b> of demand with <b>${fmtMoney(NV.val)}</b> of stock and <b>${fmtInt(NV.below)}</b> products below 95% in-stock (TSO-DTF: <b>${fmtInt(T.below)}</b>).`;
    if (vMA) call += ` The 7-day-average rule needs <b>${fmtMoney(vMA)}</b> for the TSO-DTF fill rate, ${pct((vMA - T.val) / vMA)} more stock.`;
    document.getElementById("inv-callout").innerHTML = call;
    document.getElementById("inv-tiles").innerHTML = [
      [pct(T.fill), "Fill rate", "share of demand served from stock", "good"],
      [pct(T.svc), "In-stock days", "share of product-days with no stock-out"],
      [fmtMoney(T.val), "Average stock value", `${fmtInt(T.inv)} units on shelves`],
      [fmtMoney(T.lval), "Lost sales", `${fmtInt(T.lost)} units over 48 days`, "bad"],
      [fmtInt(T.below), "Products below 95% in-stock", `of ${fmtInt(T.n)} (${pct(T.below / (T.n || 1))})`, T.below / (T.n || 1) > 0.15 ? "warn" : ""]
    ].map(([b, l, s, c]) => `<div class="tile"><div class="lbl">${l}</div><div class="big ${c || ""}">${b}</div><div class="sub muted">${s}</div></div>`).join("");
    // comparison table
    const rowsC = [["tso", T], ["base", NV], ["ma7", MA]];
    document.getElementById("cmp-sub").textContent = `Same ${pct(sl, sl * 100 % 1 ? 1 : 0)} service target and safety-stock method for every rule; only the forecast differs.`;
    document.getElementById("t-cmp").innerHTML = `<thead><tr><th>Forecast</th><th>Fill rate</th><th>Stock value</th><th>Lost sales</th><th>Below 95%</th></tr></thead><tbody>` +
      rowsC.map(([k, r]) => `<tr class="${k === "tso" ? "hl" : ""}"><td>${POL[k]}</td><td class="num">${pct(r.fill, 2)}</td><td class="num">${fmtMoney(r.val)}</td><td class="num">${fmtMoney(r.lval)}</td><td class="num">${fmtInt(r.below)}</td></tr>`).join("") + "</tbody>";
    // trade-off chart
    const el = document.getElementById("c-trade"), W = Math.max(300, el.clientWidth), H = W < 520 ? 250 : 300, pad = { l: 58, r: 14, t: 12, b: 38 };
    const curves = ["tso", "base", "ma7"].map((m, k) => ({ m, c: [css("--s1"), css("--s2"), css("--s6")][k], pts: SLS.map((s, i) => ({ ...invAgg(m, s), i })) }));
    const all = curves.flatMap(c => c.pts), fx0 = Math.min(...all.map(p => p.fill)), fx1 = Math.max(...all.map(p => p.fill));
    const x0 = Math.floor(fx0 * 100) / 100, x1 = Math.ceil(fx1 * 100) / 100, [top, yt] = nice(Math.max(...all.map(p => p.val)) * 1.05);
    const sx = v => pad.l + (W - pad.l - pad.r) * (v - x0) / (x1 - x0), sy = v => pad.t + (H - pad.t - pad.b) * (1 - v / top);
    let s = yt.map(t => `<line x1="${pad.l}" x2="${W - pad.r}" y1="${sy(t)}" y2="${sy(t)}" stroke="${css("--line-2")}"/><text x="${pad.l - 8}" y="${sy(t) + 4}" text-anchor="end" class="t-num">$${short(t)}</text>`).join("");
    const xstep = (x1 - x0) > 0.06 ? 0.02 : 0.01;
    for (let v = x0; v <= x1 + 1e-9; v += xstep) s += `<text x="${sx(v)}" y="${H - 20}" text-anchor="middle" class="t-num">${Math.round(v * 100)}%</text>`;
    s += `<text x="${(pad.l + W - pad.r) / 2}" y="${H - 3}" text-anchor="middle" style="font-size:11px">Fill rate (share of demand served)</text>`;
    for (const c of curves) {
      s += `<polyline points="${c.pts.map(p => `${sx(p.fill)},${sy(p.val)}`).join(" ")}" fill="none" stroke="${c.c}" stroke-width="2" ${c.m === "tso" ? "" : 'stroke-dasharray="5 4"'}/>`;
      c.pts.forEach(p => { const on = p.i === state.sl; s += `<circle cx="${sx(p.fill)}" cy="${sy(p.val)}" r="${on ? 7 : 4.5}" fill="${on ? c.c : css("--panel")}" stroke="${c.c}" stroke-width="2" class="pt" data-m="${c.m}" data-i="${p.i}"/>`; });
    }
    el.innerHTML = `<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Stock value against fill rate for each ordering rule">${s}</svg>`;
    el.querySelectorAll(".pt").forEach(ci => { const c = curves.find(x => x.m === ci.dataset.m), p = c.pts[+ci.dataset.i];
      ci.addEventListener("mousemove", ev => showTip(`<b>${POL[c.m]}</b> · target ${slItems[p.i][1]}<br>Fill rate <span class="n">${pct(p.fill)}</span><br>Avg stock value <span class="n">${fmtMoney(p.val)}</span><br>Lost sales <span class="n">${fmtMoney(p.lval)}</span>`, ev)); ci.addEventListener("mouseleave", hideTip); });
    document.getElementById("trade-legend").innerHTML = curves.map(c => `<span><i class="${c.m === "tso" ? "line" : "dash"}" style="background:${c.c};border-color:${c.c}"></i>${POL[c.m]}</span>`).join("") + `<span class="muted">Large dot: chosen target</span>`;
    // by category
    const cats = CATS.filter(c => state.cat < 0 || c === CATS[state.cat]);
    document.getElementById("t-invcat").innerHTML = `<thead><tr><th>Category</th><th>Products</th><th>Fill rate</th><th>In-stock days</th><th>Avg stock value</th><th>Lost sales</th><th>Below 95% in-stock</th></tr></thead><tbody>` +
      cats.map(c => { const t = { dem: 0, sold: 0, val: 0, lval: 0, slsum: 0, below: 0, n: 0 }; D.inv.forEach(r => { if (r[0] !== "tso" || r[1] !== sl || r[3] !== c || !keep(r[2], r[3], r[4])) return; t.dem += r[5]; t.sold += r[6]; t.val += r[9]; t.lval += r[10]; t.slsum += r[11]; t.below += r[12]; t.n += r[13]; });
        return t.n ? `<tr><td>${CAT_LABEL[c]}</td><td class="num">${fmtInt(t.n)}</td><td class="num">${pct(t.sold / t.dem)}</td><td class="num">${pct(t.slsum / t.n)}</td><td class="num">${fmtMoney(t.val)}</td><td class="num">${fmtMoney(t.lval)}</td><td class="num">${fmtInt(t.below)} (${pct(t.below / t.n, 0)})</td></tr>` : ""; }).join("") + "</tbody>";
  }

  // ---------- ORDER PLAN ----------
  const Z = [0.8416, 1.0364, 1.2816, 1.6449, 1.9600, 2.3263];
  function planRows() {
    const z = Z[state.sl];
    return D.plan.filter(p => keep(p[0], p[1].split("_")[0], p[3])).map(p => {
      const ss = z * (p[6] || 0), S = p[5] + ss, oh = p[8 + state.sl] || 0, q = Math.max(0, Math.ceil(S - oh - 1e-9));
      return { store: p[0], item: p[1], dept: p[2], type: p[3], price: p[4] || 0, fc: p[5], ss, S, oh, q, val: q * (p[4] || 0), act: p[7] };
    });
  }
  function renderOrder() {
    const all = planRows();
    let rows = state.q ? all.filter(r => r.item.includes(state.q) || r.dept.includes(state.q)) : all;
    const k = state.sortKey, d = state.sortDir; rows = rows.slice().sort((a, b) => (typeof a[k] === "string" ? a[k].localeCompare(b[k]) : a[k] - b[k]) * d);
    const nOrd = all.filter(r => r.q > 0).length, units = all.reduce((t, r) => t + r.q, 0), val = all.reduce((t, r) => t + r.val, 0), fcu = all.reduce((t, r) => t + r.fc, 0);
    document.getElementById("order-title").textContent = `Order plan for ${fmtDate(M.planDate)} 2016`;
    document.getElementById("order-tiles").innerHTML = [
      [fmtInt(nOrd), "Products to reorder", `of ${fmtInt(all.length)} in view`],
      [fmtInt(units), "Units to order", `forecast demand ${fmtInt(fcu)} units`],
      [fmtMoney(val), "Order value", "at current selling prices"],
      [slItems[state.sl][1], "Service target", "change it above the table"]
    ].map(([b, l, s]) => `<div class="tile"><div class="lbl">${l}</div><div class="big">${b}</div><div class="sub muted">${s}</div></div>`).join("");
    const cols = [["item", "Product"], ["store", "Store"], ["type", "Demand type"], ["price", "Price"], ["fc", "Forecast"], ["ss", "Safety stock"], ["S", "Order-up-to"], ["oh", "On hand"], ["q", "Order qty"], ["val", "Order value"]];
    const show = rows.slice(0, state.shown);
    document.getElementById("t-order").innerHTML = `<thead><tr>${cols.map(([c, l]) => `<th class="sortable" data-k="${c}" aria-sort="${c === k ? (d > 0 ? "ascending" : "descending") : "none"}">${l}${c === k ? (d > 0 ? " ▲" : " ▼") : ""}</th>`).join("")}</tr></thead><tbody>` +
      (show.length ? show.map(r => `<tr><td class="mono">${r.item}</td><td>${r.store}</td><td style="text-align:right"><span class="pill ${r.type}">${TYPE_LABEL[r.type]}</span></td><td class="num">$${r.price.toFixed(2)}</td><td class="num">${r.fc.toFixed(1)}</td><td class="num">${r.ss.toFixed(1)}</td><td class="num">${r.S.toFixed(1)}</td><td class="num">${fmtInt(r.oh)}</td><td class="num">${r.q ? `<span class="pill order">${fmtInt(r.q)}</span>` : `<span class="pill none">0</span>`}</td><td class="num">${fmtMoney(r.val)}</td></tr>`).join("")
        : `<tr><td colspan="10" class="muted">No product matches “${esc(state.q)}”. Check the spelling or clear the filters.</td></tr>`) + "</tbody>";
    document.querySelectorAll("#t-order th.sortable").forEach(th => th.addEventListener("click", () => { const kk = th.dataset.k; if (state.sortKey === kk) state.sortDir *= -1; else { state.sortKey = kk; state.sortDir = typeof rows[0]?.[kk] === "string" ? 1 : -1; } renderOrder(); }));
    const more = document.getElementById("order-more"); more.hidden = rows.length <= state.shown; more.textContent = `Show 25 more (${fmtInt(rows.length - Math.min(rows.length, state.shown))} left)`;
  }

  // ---------- MODEL ----------
  function renderModel() {
    const X = D.model, imp = v => `<span class="${v > 0.005 ? "good" : v < -0.005 ? "bad" : ""}">${sgn(v, 2)}</span>`;
    const off = X.levels[X.levels.length - 1];
    document.getElementById("model-tiles").innerHTML = [
      [sgn(off[3], 2), "Official M5 score improvement", `${off[1].toFixed(4)} → ${off[2].toFixed(4)}`, "good"],
      [`${X.ci[0].toFixed(2)}% – ${X.ci[1].toFixed(2)}%`, "95% bootstrap interval", `${X.ci[2].toFixed(0)}% of resamples above 1%`],
      [pct(X.improved / 100), "Products more accurate", `Wilcoxon p = ${X.p.toExponential(1)}`],
      [X.acc.map(v => pct(v)).join(" → "), "Product accuracy (1 − WAPE)", "base paper DTF → TSO-DTF"]
    ].map(([b, l, s, c]) => `<div class="tile"><div class="lbl">${l}</div><div class="big ${c || ""}">${b}</div><div class="sub muted">${s}</div></div>`).join("");
    hbars(document.getElementById("c-levels"), X.levels.map(r => ({ label: r[0].replace(" (12 levels)", ""), v: Math.max(r[3], 0), c: r[0].startsWith("Official") ? css("--s3") : css("--s1"), txt: sgn(r[3], 2),
      tip: `<b>${r[0]}</b><br>Base paper DTF <span class="n">${r[1].toFixed(4)}</span><br>TSO-DTF <span class="n">${r[2].toFixed(4)}</span>` })), { lw: 150, rw: 70, label: "Improvement by level" });
    const tbl = (rows, hl) => `<thead><tr><th>${hl}</th><th>Base paper DTF</th><th>TSO-DTF</th><th>Change</th></tr></thead><tbody>` +
      rows.map(r => `<tr class="${r[0].startsWith("Official") ? "hl" : ""}"><td>${r[0]}</td><td class="num">${r[1].toFixed(4)}</td><td class="num">${r[2].toFixed(4)}</td><td class="num">${imp(r[3])}</td></tr>`).join("") + "</tbody>";
    document.getElementById("t-levels").innerHTML = tbl(X.levels, "Level");
    document.getElementById("t-metrics").innerHTML = tbl(X.metrics, "Metric");
    document.getElementById("t-model").innerHTML = `<thead><tr><th>Model</th><th>Product WRMSSE</th><th>Change</th><th>Official M5</th><th>Change</th></tr></thead><tbody>` +
      X.abl.map(r => `<tr class="${r[0].startsWith("TSO") ? "hl" : ""}"><td>${r[0]}</td><td class="num">${r[1].toFixed(4)}</td><td class="num">${r[0].startsWith("Base") ? "–" : imp(r[2])}</td><td class="num">${r[3].toFixed(4)}</td><td class="num">${r[0].startsWith("Base") ? "–" : imp(r[4])}</td></tr>`).join("") + "</tbody>";
  }

  // ---------- render ----------
  function render() {
    ["f-store", "f-cat", "f-type", "f-sl", "f-sl2"].forEach(id => segSync(id, { "f-store": "store", "f-cat": "cat", "f-type": "type", "f-sl": "sl", "f-sl2": "sl" }[id]));
    document.querySelectorAll(".tabs button").forEach(b => { const on = b.id === "tab-" + state.tab; b.setAttribute("aria-selected", String(on)); document.getElementById(b.getAttribute("aria-controls")).hidden = !on; });
    const n = D.plan.filter(p => keep(p[0], p[1].split("_")[0], p[3])).length;
    document.getElementById("count").textContent = `${n.toLocaleString("en-US")} products`;
    hideTip();
    ({ sales: renderSales, fc: renderFc, inv: renderInv, order: renderOrder, model: renderModel })[state.tab]();
  }
  let lastW = 0; new ResizeObserver(() => { const w = document.querySelector(".wrap").clientWidth; if (Math.abs(w - lastW) > 2) { lastW = w; render(); } }).observe(document.querySelector(".wrap"));
  render();
  window.__render = t => { state.tab = t; render(); };
})();
</script>


## Step 5 - Build, save and show the dashboard

In [6]:
from IPython.display import HTML, display
js = json.dumps(DATA, separators=(",", ":"), default=float).replace("NaN", "null").replace("</", "<\\/")
page = open("dashboard_template.html").read().replace("__DATA__", js)
page = '<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"></head><body style="margin:0">' + page + "</body></html>"
out = f"{RES_DIR}/dashboard.html"; open(out, "w").write(page)
print(f"Saved {out} ({len(page) / 1e6:.1f} MB) - download it and open in Chrome/Edge to use it full screen.")
display(HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:1000px;border:1px solid #ddd"></iframe>'))


Saved /content/drive/MyDrive/m5_results/dashboard.html (1.3 MB) - download it and open in Chrome/Edge to use it full screen.
